# Setup & Configuration

In [1]:
# ─── Standard Library ───
import os
import re
import json
import time
import base64
from io import BytesIO
from pathlib import Path

# ─── Third-Party ───
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from PIL import Image
from openai import OpenAI

### API Keys & Model Configuration

In [ ]:
# ─── OpenRouter credentials ───
OPENROUTER_API_KEY = "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"
MODEL_NAME = "openrouter/stepfun/step-3.5-flash:free"

# ─── Ngrok endpoint ───
NGROK_URL = "https://3325-194-27-149-203.ngrok-free.app/v1"

# ─── Local models to benchmark ───
MODELS_TO_TEST = ["gemma4:e4b"]

# ─── Retry budget for every API call ───
MAX_RETRIES = 3

# ─── Ollama-tunnelled client ───
client = OpenAI(
    # base_url=NGROK_URL,
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=180.0
)

### Dataset Paths

In [3]:
CHARTQA_PATH    = Path("../DChartQA/chartqa.parquet")
MATHVISION_PATH = Path("../DMathVision/mathvision.parquet")
SCREENQA_PATH   = Path("../DScreenQA/screenqa.parquet")
TURTLEBENCH_PATH = Path("../DTurtleBench")
MTVQA_PATH      = Path("../DMTVQA/data/mtvqa_AR.parquet")
PEARL_PATH      = Path("../DPEARL/data/pearl.parquet")

### Shared Helper Functions

In [5]:
def encode_image_bytes_to_base64(byte_data, resize=None):
    """Convert raw image bytes to a base64 data-URL string.

    Parameters
    ----------
    byte_data : bytes
        Raw image bytes (e.g. from a parquet column).
    resize : tuple[int, int] | None
        Optional (width, height) cap; the image is thumbnailed to fit.

    Returns
    -------
    str | None
        A 'data:image/jpeg;base64,...' string, or None on failure.
    """
    try:
        img = Image.open(BytesIO(byte_data))
        if resize:
            img.thumbnail(resize)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_raw_bytes_to_base64(byte_data):
    """Encode raw bytes directly to a base64 data-URL (no PIL processing)."""
    try:
        b64 = base64.b64encode(byte_data).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_image_file_to_base64(image_path, include_prefix=True):
    """Open an image file from disk, convert to JPEG base64.

    Parameters
    ----------
    image_path : str | Path
        Path to the image on disk.
    include_prefix : bool
        If True, return with 'data:image/jpeg;base64,' prefix.

    Returns
    -------
    str | None
    """
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}" if include_prefix else b64
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None


def query_openrouter(prompt, base64_image, model_name=None, max_retries=MAX_RETRIES):
    """Send an image + prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": base64_image}},
            ],
        }],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openai_client(oai_client, model_name, prompt, base64_image, max_retries=MAX_RETRIES):
    """Send an image + prompt via an OpenAI-compatible client (Ollama / Ollama)."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": base64_image}},
        ],
    }]

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name, messages=messages,
                stream=False, temperature=0.1,
            )
            text = resp.choices[0].message.content.strip()
            return text, time.time() - start
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openai_client_text(oai_client, model_name, prompt, max_retries=MAX_RETRIES):
    """Send a text-only prompt via an OpenAI-compatible client."""
    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
            )
            return resp.choices[0].message.content.strip(), time.time() - start
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def load_progress(output_csv, key_columns):
    """Load already-processed task keys from an existing CSV to enable resuming.

    Parameters
    ----------
    output_csv : Path
        CSV file path.
    key_columns : list[str]
        Column(s) whose concatenation forms a unique task key.

    Returns
    -------
    tuple[set, bool]
        (processed_keys, file_exists)
    """
    processed = set()
    exists = output_csv.is_file()
    if exists:
        try:
            df = pd.read_csv(output_csv)
            if not df.empty and all(c in df.columns for c in key_columns):
                if len(key_columns) == 1:
                    processed = set(df[key_columns[0]].astype(str))
                else:
                    processed = set(
                        df[key_columns[0]].astype(str).str.cat(
                            [df[c].astype(str) for c in key_columns[1:]], sep="_"
                        )
                    )
                print(f"🔄 Resuming — skipping {len(processed)} already-processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV: {e}")
    return processed, exists


def append_result(result_dict, output_csv, file_exists):
    """Append a single result row to CSV, writing the header only once.

    Returns
    -------
    bool
        Updated file_exists flag (always True after first write).
    """
    pd.DataFrame([result_dict]).to_csv(
        output_csv, mode="a", header=not file_exists, index=False
    )
    return True


def safe_model_filename(name):
    """Sanitise a model name for use inside a filename."""
    return name.replace("/", "_").replace(":", "_")


def extract_image_bytes(row):
    """Extract raw image bytes from a parquet row (handles nested dict or flat column)."""
    if "decoded_image" in row and isinstance(row["decoded_image"], dict):
        return row["decoded_image"].get("bytes")
    if "image" in row and isinstance(row["image"], dict) and "bytes" in row["image"]:
        return row["image"]["bytes"]
    if "image" in row and isinstance(row["image"], bytes):
        return row["image"]
    return row.get("bytes")

# MathVision Evaluation

### Setup

In [6]:
# ─── Load MathVision dataset ───
mathvision_df = None
if MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {MATHVISION_PATH}.")

✅ Loaded 3040 rows.


### Prompt & Evaluator

In [7]:
MATHVISION_SYSTEM_PROMPT = """
You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""


def evaluate_math(ans, gt):
    """Check whether the model answer matches the ground truth (exact or substring)."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    return ans_clean == gt_clean or gt_clean in ans_clean

### OpenRouter

In [ ]:
# ─── MathVision run variables (OpenRouter) ───
MATHVISION_OR_LIMIT = 1  # Set to None for full evaluation

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run MathVision evaluation through OpenRouter API."""
    print(f"🚀 Starting MathVision OpenRouter inference — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(3)


# ─── Execute ───
if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=MATHVISION_OR_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

### Ollama

In [8]:
# ─── MathVision run variables (Ollama) ───
MATHVISION_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [10]:
def run_mathvision_ollama(df, limit=None):
    """Run MathVision evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 MathVision Ollama — {model_name}...")

        results_dir = Path("MathVisionResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mathvision_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["id"])

        count = 0
        for _, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
            task_id = str(row["id"])
            if task_id in processed:
                continue

            level   = str(row.get("level", "N/A"))
            subject = str(row.get("subject", "N/A"))
            query   = str(row["question"])
            gt      = str(row["answer"]).strip()

            print(f"\nTask {task_id} | {subject} | Lvl {level}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_math(answer, gt)

            file_exists = append_result({
                "id": task_id, "level": level, "subject": subject,
                "image": str(row.get("image", "N/A")), "question": query,
                "ground_truth": gt, "model_answer": answer,
                "evaluation_passed": passed, "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_id)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)


# ─── Execute ───
if mathvision_df is not None:
    run_mathvision_ollama(mathvision_df, limit=MATHVISION_OLLAMA_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")


🚀 MathVision Ollama — gemma4:e4b...

Task 1 | arithmetic | Lvl 2
   ✅ | 95.64s | '60'

Task 2 | arithmetic | Lvl 2
   ✅ | 29.36s | 'A'
🎯 Limit of 2 reached for gemma4:e4b.


# ChartQA Evaluation

### Setup

In [11]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(CHARTQA_PATH)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {CHARTQA_PATH}.")

✅ Loaded 32719 rows.
Columns: ['imgname', 'query', 'label', 'type', 'image']


### Prompt & Evaluator

In [12]:
CHARTQA_SYSTEM_PROMPT = """
You are a strict data extraction API analyzing a chart. Provide the exact answer to the user's question using only the visual data.

STRICT FORMATTING RULES:
1. RAW DATA ONLY: Output the final answer and nothing else. Zero conversational text, zero reasoning, and zero introductory filler (Never write "The answer is" or "Based on the chart").
2. NUMBERS: Output the exact digits. Include units, currencies, or percentages only if they are the direct answer. Do not spell out numbers as words.
3. BOOLEANS: If the question implies a true/false or yes/no answer, output strictly "Yes" or "No".
4. LISTS: If the answer requires multiple chart labels, separate them strictly with a comma and a space (e.g., "France, Germany, Italy").
5. EXACT MATCH: Copy axis labels, legend items, or categories exactly as they appear in the chart image.
"""


def evaluate_chartqa(ans, gt):
    """Relaxed match: any ground-truth label found inside the model answer."""
    ans_clean = str(ans).lower().strip()
    valid_labels = gt if isinstance(gt, list) else [gt]
    return any(str(label).lower() in ans_clean for label in valid_labels)

### OpenRouter

In [ ]:
# ─── ChartQA run variables (OpenRouter) ───
CHARTQA_OR_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter API."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["query"])
        label    = row["label"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query,
            "ground_truth": str(label), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(3)


# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=CHARTQA_OR_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

### Ollama

In [13]:
# ─── ChartQA run variables (Ollama) ───
CHARTQA_OLLAMA_LIMIT = 2 # Set to None for full evaluation

In [ ]:
def run_chartqa_ollama(df, limit=None):
    """Run ChartQA evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"🚀 ChartQA Ollama — {model_name}...")

        results_dir = Path("ChartQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"chartqa_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["imgname", "query"])

        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            imgname  = str(row["imgname"])
            query    = str(row["query"])
            label    = row["label"]
            qa_type  = str(row["type"])
            task_key = f"{imgname}_{query}"

            if task_key in processed:
                continue

            print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

            b64 = encode_image_bytes_to_base64(row["image"])
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_chartqa(answer, label)

            file_exists = append_result({
                "id": str(index), "imgname": imgname, "query": query,
                "ground_truth": str(label), "type": qa_type,
                "model_answer": answer, "evaluation_passed": passed,
                "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_key)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
            time.sleep(3)


# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_ollama(chartqa_df, limit=CHARTQA_OLLAMA_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

🚀 ChartQA Ollama — gemma4:e4b...
[0] Image: two_col_22779.png | Query: How many colleges did Bengaluru have in 2019?
   ✅ | 20.46s | '880' | GT: '880'
[1] Image: two_col_22779.png | Query: What is the capital of Rajasthan?
   ✅ | 118.08s | 'Jaipur' | GT: 'Jaipur'
🎯 Limit of 2 reached for gemma4:e4b.


# TurtleBench Evaluation

### Prompt & Evaluator

In [16]:
TURTLEBENCH_SYSTEM_PROMPT = """
INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution.
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""

TURTLEBENCH_EVAL_PROMPT_TEMPLATE = """Compare the following two Python Turtle scripts.
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer with 'Yes' or 'No' followed by one statment explaining the difference. Do not exceed one statment and do not give a long statment.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""


def evaluate_turtle_equivalence_openrouter(model_code, gt_code):
    """Use OpenRouter to judge whether two Turtle scripts produce the same output."""
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=model_code, ground_truth_code=gt_code
    )
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
    }
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            raw = resp.json()["choices"][0]["message"]["content"].strip().lower()
            if "yes" in raw:
                return True
            if "no" in raw:
                return False
            return raw
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR"


def evaluate_turtle_equivalence_client(oai_client, model_name, model_code, gt_code):
    """Use an OpenAI-compatible client to judge Turtle script equivalence."""
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=model_code, ground_truth_code=gt_code
    )
    for attempt in range(MAX_RETRIES):
        try:
            resp = oai_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
            )
            raw = resp.choices[0].message.content.strip().lower()
            if "yes" in raw:
                return True
            if "no" in raw:
                return False
            return raw
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5 * (attempt + 1))
    return "ERROR" 

### Dataset Loader

In [17]:
def load_turtle_bench(dataset_path):
    """Walk the DTurtleBench/Tasks folder tree and return a DataFrame of questions."""
    tasks_dir = Path(dataset_path) / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ Tasks folder not found at {tasks_dir}")
        return None

    rows = []
    for task_folder in sorted(tasks_dir.iterdir()):
        if not task_folder.is_dir():
            continue
        task_id    = task_folder.name
        text_dir   = task_folder / "QA" / "text"
        code_dir   = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"

        if not text_dir.exists():
            continue
        for qfile in sorted(text_dir.glob("q*.txt")):
            qid = qfile.stem
            query_text = qfile.read_text(encoding="utf-8").strip()
            code_file  = code_dir / f"{qid}_code.txt"
            code_text  = code_file.read_text(encoding="utf-8").strip() if code_file.exists() else "No code found."

            rows.append({
                "task_id": task_id, "question_id": qid,
                "query_text": query_text, "code_text": code_text,
                "task_folder": str(task_folder), "query_path": str(qfile),
                "code_path": str(code_file), "image_path": str(image_path),
            })
    return pd.DataFrame(rows)


def find_turtlebench_path():
    """Auto-detect the DTurtleBench folder by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if parent.name.lower() == "thesis":
            candidate = parent / "DTurtleBench"
            if candidate.exists():
                return candidate
    # Absolute fallback
    fallback = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
    return fallback if fallback.exists() else None

### OpenRouter

In [ ]:
# ─── TurtleBench run variables (OpenRouter) ───
TURTLEBENCH_OR_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_turtlebench_openrouter(df, limit=None):
    """Run TurtleBench evaluation through OpenRouter API."""
    print(f"🚀 TurtleBench OpenRouter — {MODEL_NAME}...")

    results_dir = Path("TurtleBenchResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"TurtleBench_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["task_id", "question_id"])

    eval_df = df.head(limit) if limit else df
    for _, row in eval_df.iterrows():
        task_key = f"{row['task_id']}_{row['question_id']}"
        if task_key in processed:
            continue

        print(f"\nTask {row['task_id']} | Q: {row['question_id']}...")

        b64 = encode_image_file_to_base64(row["image_path"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{row['query_text']}\n{TURTLEBENCH_SYSTEM_PROMPT}"
        model_code, gen_time = query_openrouter(prompt, b64)
        print(f"   ⏱️ Generated in {gen_time:.2f}s")
        time.sleep(2)

        gt_code = row["code_text"].strip()
        print("   🔍 Evaluating code equivalence...")
        passed = evaluate_turtle_equivalence_openrouter(model_code, gt_code)

        file_exists = append_result({
            "task_id": row["task_id"], "question_id": row["question_id"],
            "image_path": row["image_path"], "query_text": row["query_text"],
            "model_response": model_code, "ground_truth_code": gt_code,
            "codegeneration_time": gen_time, "evaluation_passed": passed,
        }, output_csv, file_exists)

        processed.add(task_key)
        print(f"   {'✅' if passed is True else '❌'} Done.")
        time.sleep(2)


# ─── Execute ───
_tb_path = find_turtlebench_path()
if _tb_path:
    _tb_df = load_turtle_bench(_tb_path)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_openrouter(_tb_df, limit=TURTLEBENCH_OR_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

### Ollama

In [18]:
# ─── TurtleBench run variables (Ollama) ───
TURTLEBENCH_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [19]:
def run_turtlebench_ollama(df, limit=None):
    """Run TurtleBench evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 TurtleBench Ollama — {model_name}...")

        results_dir = Path("TurtleBenchResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"turtlebench_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["task_id", "question_id"])

        eval_df = df.head(limit) if limit else df
        for _, row in eval_df.iterrows():
            task_key = f"{row['task_id']}_{row['question_id']}"
            if task_key in processed:
                continue

            print(f"\nTask {row['task_id']} | Q: {row['question_id']}...")

            # Ollama expects raw base64 without prefix
            b64 = encode_image_file_to_base64(row["image_path"], include_prefix=False)
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{row['query_text']}\n{TURTLEBENCH_SYSTEM_PROMPT}"
            b64_url = f"data:image/jpeg;base64,{b64}"
            model_code, gen_time = query_openai_client(client, model_name, prompt, b64_url)
            print(f"   ⏱️ Generated in {gen_time:.2f}s")
            time.sleep(2)

            gt_code = row["code_text"].strip()
            print("   🔍 Evaluating code equivalence...")
            passed = evaluate_turtle_equivalence_client(client, model_name, model_code, gt_code)

            file_exists = append_result({
                "task_id": row["task_id"], "question_id": row["question_id"],
                "image_path": row["image_path"], "query_text": row["query_text"],
                "model_response": model_code, "ground_truth_code": gt_code,
                "code_generation_time": gen_time, "evaluation_passed": passed,
            }, output_csv, file_exists)

            processed.add(task_key)
            print(f"   {'✅' if passed is True else '❌'} Done.")
            time.sleep(2)


# ─── Execute ───
_tb_path = find_turtlebench_path()
if _tb_path:
    _tb_df = load_turtle_bench(_tb_path)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_ollama(_tb_df, limit=TURTLEBENCH_OLLAMA_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

✅ Loaded 260 questions.

🚀 TurtleBench Ollama — gemma4:e4b...

Task 1 | Q: q1...
   ⏱️ Generated in 26.70s
   🔍 Evaluating code equivalence...
   ❌ Done.

Task 1 | Q: q2...
   ⏱️ Generated in 68.26s
   🔍 Evaluating code equivalence...
   ❌ Done.


# ScreenQA Evaluation

### Setup

In [20]:
# ─── Load ScreenQA dataset ───
screenqa_df = None
if SCREENQA_PATH.exists():
    metadata = pq.read_metadata(SCREENQA_PATH)
    print(f"📊 ScreenQA dataset has {metadata.num_rows} rows.")
    screenqa_df = pq.read_table(SCREENQA_PATH).to_pandas()
else:
    print(f"❌ Dataset not found at {SCREENQA_PATH}.")

📊 ScreenQA dataset has 86025 rows.


### Prompt & Evaluator

In [21]:
SCREENQA_SYSTEM_PROMPT = """
You are an expert visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
"""

# ─── Image resize dimensions for ScreenQA ───
SCREENQA_RESIZE = (1280, 720)


def evaluate_screenqa(ans, gt):
    """Relaxed match with currency/symbol stripping for screen-based QA."""
    ans_str = re.sub(r'[\$\£\€\,]', '', str(ans).lower())
    gt_str  = re.sub(r'[\$\£\€\,]', '', str(gt).lower())

    ans_clean = re.sub(r'[^\w]', '', ans_str)
    gt_clean  = re.sub(r'[^\w]', '', gt_str)

    if not ans_clean or not gt_clean:
        return False
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
    # Prevent single-char false positives (e.g. "1" matching "100")
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    return False


def extract_screenqa_ground_truth(row):
    """Safely extract the ground-truth answer from ScreenQA's nested structure."""
    raw = row.get("answers", row.get("ground_truth", row.get("full_answer", [])))
    try:
        return str(raw[0]["full_answer"]).strip()
    except (IndexError, KeyError, TypeError):
        return str(raw).strip()

### OpenRouter

In [ ]:
# ─── ScreenQA run variables (OpenRouter) ───
SCREENQA_OR_LIMIT = 4  # Set to None for full evaluation

In [ ]:
def run_screenqa_openrouter(df, limit=None):
    """Run ScreenQA evaluation through OpenRouter API."""
    print(f"🚀 ScreenQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ScreenQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"screenqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        task_id = str(row["screen_id"])
        if task_id in processed:
            continue

        image_name = str(row.get("file_name", "N/A"))
        query = str(row["question"])
        gt = extract_screenqa_ground_truth(row)

        print(f"\nTask {task_id} | Image: {image_name}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64, MODEL_NAME)
        passed = evaluate_screenqa(answer, gt)

        file_exists = append_result({
            "id": task_id, "image": image_name, "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(3)


# ─── Execute ───
if screenqa_df is not None:
    run_screenqa_openrouter(screenqa_df, limit=SCREENQA_OR_LIMIT)
else:
    print("❌ screenqa_df not loaded — run the Setup cell first.")

### Ollama

In [22]:
# ─── ScreenQA run variables (Ollama) ───
SCREENQA_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [23]:
def run_screenqa_ollama(df, limit=None):
    """Run ScreenQA evaluation through Ollama-tunnelled backend."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 ScreenQA Ollama — {model_name}...")

        results_dir = Path("ScreenQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"screenqa_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["id"])

        count = 0
        for _, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            task_id = str(row["screen_id"])
            if task_id in processed:
                continue

            image_name = str(row.get("file_name", "N/A"))
            query = str(row["question"])
            gt = extract_screenqa_ground_truth(row)

            print(f"\nTask {task_id} | Image: {image_name}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_screenqa(answer, gt)

            file_exists = append_result({
                "id": task_id, "image": image_name, "question": query,
                "ground_truth": gt, "model_answer": answer,
                "evaluation_passed": passed, "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(task_id)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)


# ─── Execute ───
if screenqa_df is not None:
    run_screenqa_ollama(screenqa_df, limit=SCREENQA_OLLAMA_LIMIT)
else:
    print("❌ screenqa_df not loaded — run the Setup cell first.")


🚀 ScreenQA Ollama — gemma4:e4b...

Task 16602 | Image: images/rico/16602.jpg
   ✅ | 51.55s | '$70'

Task 16603 | Image: images/rico/16603.jpg
   ✅ | 15.24s | 'Grace'
🎯 Limit of 2 reached for gemma4:e4b.


# PEARL Evaluation

### Setup

In [24]:
# ─── Load PEARL dataset ───
pearl_df = None
if PEARL_PATH.exists():
    pearl_df = pd.read_parquet(PEARL_PATH)
    print(f"✅ Loaded {len(pearl_df)} rows.")
else:
    print(f"❌ Dataset not found at {PEARL_PATH}.")

✅ Loaded 4832 rows.


### Prompt & Evaluator

In [25]:
PEARL_SYSTEM_PROMPT = (
    "You are an expert visual analyst. Answer the user's question using only the provided image and text.\n\n"
    "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
    "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
    "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
    "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
    "If the question is in Arabic, answer in Arabic. If in English, answer in English, etc.\n"
    "If it is a true and false question answer TRUE or FALSE, if it a multiple choice question give me the answer itself and NOT the letter.\n"
)


def evaluate_pearl(ans, gt):
    """Exact or substring match for PEARL answers."""
    a = str(ans).strip()
    g = str(gt).strip()
    if not a or not g:
        return False
    if a == g:
        return True
    if g in a:
        return True
    if len(a) >= 2 and a in g:
        return True
    return False

### Ollama

In [26]:
# ─── PEARL run variables (Ollama) ───
PEARL_OLLAMA_LIMIT = 2  # Set to None for full evaluation

In [ ]:
def run_pearl_ollama(df, limit=None):
    """Run PEARL evaluation through local Ollama."""
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 PEARL Ollama — {model_name}...")

        results_dir = Path("PEARLResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"pearl_ollama_{safe_model_filename(model_name)}_results.csv"
        processed, file_exists = load_progress(output_csv, ["index"])

        count = 0
        for idx, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            idx_str = str(idx)
            if idx_str in processed:
                continue

            query    = str(row["question"]).strip()
            gt       = str(row["answer"]).strip()
            category = str(row.get("category", "N/A"))
            image_id = str(row.get("image_id", "N/A"))

            print(f"\nTask {idx_str} | image_id: {image_id} | category: {category}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            prompt = f"{query}\n\n{PEARL_SYSTEM_PROMPT}"
            answer, dur = query_openai_client(client, model_name, prompt, b64)
            passed = evaluate_pearl(answer, gt)

            file_exists = append_result({
                "index": idx_str, "image_id": image_id, "category": category,
                "question": query, "ground_truth": gt,
                "model_answer": answer, "evaluation_passed": passed,
                "run_time": round(dur, 3),
            }, output_csv, file_exists)

            processed.add(idx_str)
            count += 1
            print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
            time.sleep(3)


# ─── Execute ───
if pearl_df is not None:
    run_pearl_ollama(pearl_df, limit=PEARL_OLLAMA_LIMIT)
else:
    print("❌ pearl_df not loaded — run the Setup cell first.")


🚀 PEARL Ollama — gemma4:e4b...

Task 0 | image_id: 131625 | category: LandMarks
   ❌ | 140.63s | 'كورنيش'

Task 1 | image_id: 32236 | category: Architecture
   ❌ | 87.79s | 'صحيح'
🎯 Limit of 2 reached for gemma4:e4b.


# Results Evaluation

In [28]:
# ─── Folders to scan for result CSVs ───
RESULTS_FOLDERS = [
    "MathVisionResults", "ChartQAResults", "ScreenQAResults",
    "TurtleBenchResults", "MTVQAResults", "PEARLResults",
]

In [29]:
def summarize_results(folders=RESULTS_FOLDERS):
    """Print a table of accuracy percentages for every result CSV found."""
    print(f"{'Model Name':<60} | {'Accuracy':>10}")
    print("-" * 75)

    for folder in folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
        for csv_file in sorted(folder_path.glob("*.csv")):
            try:
                df = pd.read_csv(csv_file)
                if "evaluation_passed" not in df.columns or len(df) == 0:
                    continue
                correct  = df["evaluation_passed"].astype(int).sum()
                accuracy = (correct / len(df)) * 100
                name = csv_file.stem.replace("_ollama", "").replace("_results", "")
                print(f"{name:<60} | {accuracy:>8.2f}%")
            except Exception as e:
                print(f"⚠️ Error processing {csv_file.name}: {e}")


summarize_results()

Model Name                                                   |   Accuracy
---------------------------------------------------------------------------
mathvision_gemma4_e4b                                        |   100.00%
mathvision_llava_7b                                          |    50.00%
mathvision_qwen3-vl_8b                                       |   100.00%
chartqa_gemma4_e4b                                           |   100.00%
chartqa_llava_7b                                             |     0.00%
chartqa_qwen3-vl_4b                                          |    75.00%
chartqa_openrouter_openrouter_healer-alpha                   |   100.00%
screenqa_gemma4_e4b                                          |   100.00%
screenqa_llava_7b                                            |     0.00%
screenqa_qwen3-vl_4b                                         |    83.33%
screenqa_openrouter_openrouter_healer-alpha                  |    75.00%
TurtleBench_openrouter_healer-alpha            